# COSC 2671 — Assignment 2
## Notebook 2: Data Cleaning
Flatten raw JSON → clean CSVs ready for network and NLP analysis.
One output CSV per repo, containing both issues and comments as rows.

***Cell 2 — Set Working Directory***

In [1]:
import os

# Always run from project root regardless of where the notebook lives
os.chdir(os.path.dirname(os.path.abspath("__file__")) + "/..")

# Confirm
print(f"Working directory: {os.getcwd()}")
print(f"data/raw exists: {os.path.exists('data/raw')}")

Working directory: /Users/rob/Desktop/2026/social/SMNA2026-A2-github-networks
data/raw exists: True


***Cell 3 Imports***


In [10]:
import re
import json
import pandas as pd
from tqdm import tqdm

RAW_DIR  = "data/raw"
PROC_DIR = "data/processed"
os.makedirs(PROC_DIR, exist_ok=True)

REPOS = [
    "scikit-learn/scikit-learn",
    "huggingface/transformers",
    "pytorch/pytorch",
    "keras-team/keras",
    "langchain-ai/langchain"
]

print("Imports OK. Output directory ready.")

Imports OK. Output directory ready.


***Cell 4 Text Cleaning Function***


In [6]:
def clean_text(text):
    """
    Clean raw GitHub issue/comment text for NLP.
    Removes: code blocks, URLs, markdown formatting, excess whitespace.
    """
    if not text or not isinstance(text, str):
        return ""
    
    # Remove fenced code blocks (``` ... ```)
    text = re.sub(r"```[\s\S]*?```", " ", text)
    
    # Remove inline code (`code`)
    text = re.sub(r"`[^`]*`", " ", text)
    
    # Remove URLs
    text = re.sub(r"https?://\S+", " ", text)
    
    # Remove markdown headers (## Header)
    text = re.sub(r"#+\s", " ", text)
    
    # Remove markdown bold/italic (**text**, *text*, __text__, _text_)
    text = re.sub(r"[*_]{1,2}([^*_]+)[*_]{1,2}", r"\1", text)
    
    # Remove HTML tags
    text = re.sub(r"<[^>]+>", " ", text)
    
    # Remove GitHub mentions (@username) — keep them as node identifiers elsewhere
    # but strip from NLP text
    text = re.sub(r"@\w+", " ", text)
    
    # Remove image/link markdown: ![alt](url) and [text](url)
    text = re.sub(r"!\[.*?\]\(.*?\)", " ", text)
    text = re.sub(r"\[.*?\]\(.*?\)", " ", text)
    
    # Remove special characters except basic punctuation
    text = re.sub(r"[^\w\s.,!?;:()\-']", " ", text)
    
    # Collapse whitespace
    text = re.sub(r"\s+", " ", text).strip()
    
    return text.lower()

***Cell 4 - Flatten Function***

In [7]:
def flatten_repo(filepath):
    """
    Load a raw JSON file and flatten into a single DataFrame.
    Each row is either an issue or a comment.
    
    Key columns:
    - id:         unique string ID (issue_123 or comment_456)
    - type:       'issue' or 'comment'
    - author:     GitHub username
    - text:       raw text
    - clean_text: cleaned text for NLP
    - state:      'open' or 'closed' (inherited by comments from their issue)
    - created_at: timestamp
    - parent_id:  for comments — the ID of the parent issue row
    - parent_author: for comments — the author of the parent issue
    - labels:     comma-separated label names (issues only)
    - issue_number: the GitHub issue number
    """
    with open(filepath, encoding="utf-8") as f:
        issues = json.load(f)
    
    rows = []
    
    for issue in issues:
        issue_id     = f"issue_{issue['issue_number']}"
        issue_author = issue.get("author")
        raw_text     = (issue.get("title") or "") + " " + (issue.get("body") or "")
        
        # Issue row
        rows.append({
            "id":             issue_id,
            "type":           "issue",
            "author":         issue_author,
            "text":           raw_text[:5000],       # cap length
            "clean_text":     clean_text(raw_text),
            "state":          issue.get("state"),
            "created_at":     issue.get("created_at"),
            "closed_at":      issue.get("closed_at"),
            "parent_id":      None,
            "parent_author":  None,
            "labels":         ", ".join(issue.get("labels", [])),
            "issue_number":   issue["issue_number"],
            "comment_count":  issue.get("comment_count", 0)
        })
        
        # Comment rows — inherit issue state, link to parent
        for comment in issue.get("comments", []):
            comment_author = comment.get("author")
            comment_body   = comment.get("body") or ""
            
            rows.append({
                "id":             f"comment_{comment['comment_id']}",
                "type":           "comment",
                "author":         comment_author,
                "text":           comment_body[:3000],
                "clean_text":     clean_text(comment_body),
                "state":          issue.get("state"),
                "created_at":     comment.get("created_at"),
                "closed_at":      None,
                "parent_id":      issue_id,
                "parent_author":  issue_author,
                "labels":         "",
                "issue_number":   issue["issue_number"],
                "comment_count":  0
            })
    
    df = pd.DataFrame(rows)
    df["created_at"] = pd.to_datetime(df["created_at"], errors="coerce", utc=True)
    
    return df

***Cell 5 Run Cleaning***

In [16]:
repo_stats = []

for repo_name in REPOS:
    safe_name   = repo_name.replace("/", "_")
    input_path  = os.path.join(RAW_DIR, f"{safe_name}_issues.json")
    output_path = os.path.join(PROC_DIR, f"{safe_name}_clean.csv")
    
    if not os.path.exists(input_path):
        print(f"⚠️  Skipping {repo_name} — raw file not found")
        continue
    
    print(f"Cleaning: {repo_name} ...", end=" ")
    df = flatten_repo(input_path)
    
    # Drop rows with no author (bots or deleted accounts)
    before = len(df)
    df = df[df["author"].notna()].copy()
    
    # Drop rows with empty clean text
    df = df[df["clean_text"].str.len() > 10].copy()
    after = len(df)
    
    df.to_csv(output_path, index=False)
    
    issues_df   = df[df["type"] == "issue"]
    comments_df = df[df["type"] == "comment"]
    
    stats = {
        "repo":          repo_name,
        "total_rows":    after,
        "issues":        len(issues_df),
        "comments":      len(comments_df),
        "unique_users":  df["author"].nunique(),
        "open_issues":   (issues_df["state"] == "open").sum(),
        "closed_issues": (issues_df["state"] == "closed").sum(),
        "date_min":      str(df["created_at"].min())[:10],
        "date_max":      str(df["created_at"].max())[:10],
        "dropped_rows":  before - after
    }
    repo_stats.append(stats)
    print(f"✅  {after:,} rows saved → {output_path}")

# Summary table — save this for your report Section 3
stats_df = pd.DataFrame(repo_stats)
stats_df.to_csv("outputs/tables/data_summary.csv", index=False)
os.makedirs("outputs/tables", exist_ok=True)
print("\n\nDATA SUMMARY TABLE")
print(stats_df.to_string(index=False))

Cleaning: scikit-learn/scikit-learn ... ✅  11,924 rows saved → data/processed/scikit-learn_scikit-learn_clean.csv
Cleaning: huggingface/transformers ... ✅  9,881 rows saved → data/processed/huggingface_transformers_clean.csv
Cleaning: pytorch/pytorch ... ✅  6,746 rows saved → data/processed/pytorch_pytorch_clean.csv
Cleaning: keras-team/keras ... ✅  11,205 rows saved → data/processed/keras-team_keras_clean.csv
Cleaning: langchain-ai/langchain ... ✅  8,323 rows saved → data/processed/langchain-ai_langchain_clean.csv


DATA SUMMARY TABLE
                     repo  total_rows  issues  comments  unique_users  open_issues  closed_issues   date_min   date_max  dropped_rows
scikit-learn/scikit-learn       11924    1996      9928          1860          428           1568 2023-04-23 2026-05-13           275
 huggingface/transformers        9881    1993      7888          2378          250           1743 2025-04-02 2026-05-14           405
          pytorch/pytorch        6746    1998      4748 

***Cell 6 — Bot Detection***

In [25]:
def is_bot(username):
    u = username.lower()
    return bool(re.search(r'(\[bot\]$|-bot$|^bot-|bot\[)', u)) or \
           u in {"pytorchbot", "scikit-learn-bot", "dosubot", "langchain-automated-triage",
                 "github-actions", "dependabot", "stale", "codecov", "google-ml-butler",
                 "pytorch-bot", "langchain-automated-triage"}

print("Checking for bots in each repo:\n")
all_bots = set()
for repo_name in REPOS:
    safe_name = SAFE_NAMES[repo_name]
    path      = f"data/processed/{safe_name}_clean.csv"
    if not os.path.exists(path):
        print(f"  ⚠️  File not found: {path}")
        continue
    df    = pd.read_csv(path)
    users = df["author"].dropna().unique()
    bots  = [u for u in users if is_bot(u)]
    if bots:
        print(f"  {repo_name}")
        for b in sorted(bots):
            count = (df["author"] == b).sum()
            print(f"    ⚠️  {b:35} — {count} posts")
        all_bots.update(bots)
    else:
        print(f"  ✅ {repo_name} — no bots found")

print(f"\nBots to remove: {all_bots}")

# Sanity check — print accounts that were previously flagged but now kept
previously_flagged = {"ParagEkbote", "YanivBohbot", "tharindubotcalm"}
rescued = [u for u in previously_flagged if not is_bot(u)]
if rescued:
    print(f"\n✅ Rescued from false-positive removal: {rescued}")

Checking for bots in each repo:

  scikit-learn/scikit-learn
    ⚠️  github-actions[bot]                 — 16 posts
    ⚠️  mihara-bot                          — 1 posts
    ⚠️  pranav-bot                          — 2 posts
    ⚠️  scikit-learn-bot                    — 436 posts
  huggingface/transformers
    ⚠️  github-actions[bot]                 — 799 posts
  pytorch/pytorch
    ⚠️  claude[bot]                         — 5 posts
    ⚠️  github-actions[bot]                 — 105 posts
    ⚠️  pytorch-bot[bot]                    — 1527 posts
    ⚠️  pytorchbot                          — 79 posts
  keras-team/keras
    ⚠️  github-actions[bot]                 — 817 posts
    ⚠️  google-labs-jules[bot]              — 6 posts
    ⚠️  google-ml-butler[bot]               — 1055 posts
  langchain-ai/langchain
    ⚠️  Euni-Bot                            — 1 posts
    ⚠️  aiqing20230305-bot                  — 1 posts
    ⚠️  bobrenze-bot                        — 16 posts
    ⚠️  dosubot[bot]   

***Cell 7  Bot Detection and Resave***

In [26]:
if not all_bots:
    print("No bots found — CSVs unchanged.")
else:
    print(f"Removing {len(all_bots)} bot account(s): {all_bots}\n")

    for repo_name in REPOS:
        safe_name = SAFE_NAMES[repo_name]
        path      = f"data/processed/{safe_name}_clean.csv"

        if not os.path.exists(path):
            continue

        df     = pd.read_csv(path)
        before = len(df)
        df     = df[~df["author"].isin(all_bots)].copy()
        after  = len(df)
        df.to_csv(path, index=False)

        print(f"  ✅ {repo_name:<35} removed {before - after:>4} rows → {after:,} remaining")

    print("\nAll CSVs re-saved without bots.")

Removing 16 bot account(s): {'langcarl[bot]', 'pytorch-bot[bot]', 'pytorchbot', 'mihara-bot', 'aiqing20230305-bot', 'dosubot[bot]', 'bobrenze-bot', 'langchain-automated-triage[bot]', 'scikit-learn-bot', 'google-labs-jules[bot]', 'claude[bot]', 'open-swe[bot]', 'Euni-Bot', 'github-actions[bot]', 'pranav-bot', 'google-ml-butler[bot]'}

  ✅ scikit-learn/scikit-learn           removed  455 rows → 11,469 remaining
  ✅ huggingface/transformers            removed  799 rows → 9,082 remaining
  ✅ pytorch/pytorch                     removed 1716 rows → 5,030 remaining
  ✅ keras-team/keras                    removed 1878 rows → 9,327 remaining
  ✅ langchain-ai/langchain              removed  665 rows → 7,658 remaining

All CSVs re-saved without bots.


***Cell 8 Final Verification***

In [28]:
print("FINAL VERIFICATION")
print("=" * 72)
print(f"{'Repo':<35} {'Total':>7} {'Issues':>8} {'Comments':>10} {'Users':>7}")
print("-" * 72)

for repo_name in REPOS:
    safe_name = SAFE_NAMES[repo_name]
    path      = f"data/processed/{safe_name}_clean.csv"

    if not os.path.exists(path):
        print(f"  ❌  {repo_name}")
        continue

    df = pd.read_csv(path)
    print(f"  {repo_name:<35} {len(df):>7,} "
          f"{(df['type']=='issue').sum():>8,} "
          f"{(df['type']=='comment').sum():>10,} "
          f"{df['author'].nunique():>7,}")

print("\nAll files verified. Notebook 2 complete ")

FINAL VERIFICATION
Repo                                  Total   Issues   Comments   Users
------------------------------------------------------------------------
  scikit-learn/scikit-learn            11,469    1,729      9,740   1,856
  huggingface/transformers              9,082    1,993      7,089   2,377
  pytorch/pytorch                       5,030    1,726      3,304     903
  keras-team/keras                      9,327    1,996      7,331   1,655
  langchain-ai/langchain                7,658    1,993      5,665   2,949

All files verified. Notebook 2 complete 
